# requires-grad-leaf-assert — worked example 1: Assert That a Tensor Is a Leaf with requires_grad Before Optimizing

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-leaf-assert`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Before passing tensors to an optimizer, you should verify two properties: (1) the tensor is a leaf node (not the result of an operation), and (2) it has `requires_grad=True`. If either check fails, the optimizer will silently skip updating the tensor — the loss will decrease but the parameters won't change. A simple assert with a descriptive message catches this bug immediately.

## Worked solution

**Step 1 — understand `.is_leaf`.** A tensor is a leaf if it was created directly (e.g., `t.randn(...)`) rather than as the result of an operation on other tensors. `nn.Parameter` tensors are always leaves.

**Step 2 — understand `requires_grad`.** Setting `requires_grad=True` tells PyTorch to track this tensor's gradient. Without it, `.backward()` won't accumulate a `.grad` on the tensor.

**Step 3 — write the assertion.** Check both properties in sequence. If `is_leaf` fails, the tensor is an op output. If `requires_grad` fails, it was created without gradient tracking or was detached.

**Step 4 — include shape in the error message.** `tuple(p.shape)` in the message helps identify which parameter is problematic when there are many.

**Step 5 — demonstrate with a mix of valid and invalid tensors.** Create a leaf with grad, a non-leaf, and a leaf without grad. Show what happens for each.

In [ ]:
import torch as t
import torch.nn as nn

def assert_optim_ready(p: t.Tensor) -> None:
    """Assert p is safe to pass to an optimizer."""
    assert p.is_leaf, (
        f'Tensor shape={tuple(p.shape)} is non-leaf. Optimizer will silently skip it.'
    )
    assert p.requires_grad, (
        f'Tensor shape={tuple(p.shape)} has requires_grad=False. No gradient, no update.'
    )

# --- exercise and print ---
t.manual_seed(0)

# Case 1: valid nn.Parameter (leaf + requires_grad)
param = nn.Parameter(t.randn(3, 4))
try:
    assert_optim_ready(param)
    print('param: PASS  (leaf=True, requires_grad=True)')
except AssertionError as e:
    print('param: FAIL -', e)

# Case 2: non-leaf (result of an op)
a = t.randn(3, requires_grad=True)
b = a * 2   # b is non-leaf
try:
    assert_optim_ready(b)
    print('b (non-leaf): PASS')
except AssertionError as e:
    print('b (non-leaf): FAIL -', e)

# Case 3: leaf but no grad
c = t.randn(3)
try:
    assert_optim_ready(c)
    print('c (no grad): PASS')
except AssertionError as e:
    print('c (no grad): FAIL -', e)